In [1]:
import numpy as np
import pandas as pd
from casadi import *
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *
from scipy.interpolate import CubicSpline
import mosek

def RaceTrackOffset(x, y, d=2.0):
    """
    Gera uma nova pista paralela onde cada ponto (x2, y2) dista exatamente
    'd' metros do ponto correspondente (x, y) original.
    
    d > 0: Desloca para um lado (ex: direita/fora)
    d < 0: Desloca para o lado oposto (ex: esquerda/dentro)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    
    # Vetores para armazenar as direções normais
    nx = np.zeros(n)
    ny = np.zeros(n)
    
    # Calcula os vetores normais para cada ponto da pista
    for i in range(n):
        # Definição dos pontos vizinhos para cálculo da tangente local
        if i == 0:
            # Primeiro ponto: usa o próximo
            tx = x[1] - x[0]
            ty = y[1] - y[0]
        elif i == n - 1:
            # Último ponto: usa o anterior
            tx = x[n-1] - x[n-2]
            ty = y[n-1] - y[n-2]
        else:
            # Pontos intermediários: diferença centralizada para maior precisão
            tx = x[i+1] - x[i-1]
            ty = y[i+1] - y[i-1]
            
        # Magnitude (comprimento) do vetor tangente
        norma = np.sqrt(tx**2 + ty**2)
        if norma == 0:
            norma = 1e-6
            
        # Vetor tangente unitário
        tx /= norma
        ty /= norma
        
        # Rotaciona o vetor tangente em 90 graus para obter a normal unitária (nx, ny)
        nx[i] = -ty
        ny[i] = tx
        
    # Aplica o deslocamento exato de 'd' metros
    x_history2 = x + d * nx
    y_history2 = y + d * ny
    
    return x_history2, y_history2

1. LOAD RACETRACK DATA & SPATIAL PARAMETRIZATION

In [23]:
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:]*size
y_mid = df['y_coords'].values[:]*size
#x_mid,y_mid = interpolar_racetrack(x_mid,y_mid)

dt = 0.1  # Control & Simulation evaluation step size (100ms)

# Ensure the track loops closed cleanly
if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

# Calculate cumulative distance (arc length 's') along track coordinates
dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

# Linear interpolators to find exact continuous (x, y) coordinates for any distance s
get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

2. CASADI OPTI PROBLEM CONFIGURATION (Defined ONCE outside loop)


In [58]:
n_horizon = 30
l = 2.5 # Wheelbase

opti = Opti()

# Decision variables over the prediction horizon
X = opti.variable(4, n_horizon + 1) # States: [x_pos, y_pos, v, psi]
U = opti.variable(2, n_horizon)     # Controls: [a, delta]

# Parameters
X0 = opti.parameter(4)
X_REF = opti.parameter(n_horizon)
Y_REF = opti.parameter(n_horizon)
V_REF = opti.parameter(n_horizon)

obj = 0
opti.subject_to(X[:, 0] == X0)

# Build the execution graph symbolic tree once
for k in range(n_horizon):
    x_k   = X[0, k]
    y_k   = X[1, k]
    v_k   = X[2, k]
    psi_k = X[3, k]
    
    a_k     = U[0, k]
    delta_k = U[1, k]

    beta = atan(0.5 * tan(delta_k))
    
    next_x   = x_k + dt * (v_k * cos(psi_k + beta))
    next_y   = y_k + dt * (v_k * sin(psi_k + beta))
    next_v   = v_k + dt * a_k
    next_psi = psi_k + dt * ((v_k / l) * sin(beta))

    opti.subject_to(X[0, k+1] == next_x)
    opti.subject_to(X[1, k+1] == next_y)
    opti.subject_to(X[2, k+1] == next_v)
    opti.subject_to(X[3, k+1] == next_psi)

    # Stage cost
    obj += (x_k - X_REF[k])**2 + (y_k - Y_REF[k])**2 + 50.0 * (v_k - V_REF[k])**2
    obj += 80 * a_k**2 + 80 * delta_k**2

# CRITICAL FIX: Replaced [-1] negative indexing with explicit [n_horizon - 1]
obj += (X[0, n_horizon] - X_REF[n_horizon - 1])**2 + \
       (X[1, n_horizon] - Y_REF[n_horizon - 1])**2 + \
       5.0 * (X[2, n_horizon] - V_REF[n_horizon - 1])**2

opti.minimize(obj)

# Bounds
opti.subject_to(opti.bounded(-10.0, U[0, :], 5))                 
opti.subject_to(opti.bounded(-np.deg2rad(30), U[1, :], np.deg2rad(30))) 
opti.subject_to(opti.bounded(0.0, X[2, :], 20))                  

# Setup solver configuration options once
opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.warm_start_init_point": "yes"}
opti.solver('ipopt', opts)


3. CLOSED-LOOP SIMULATION LOOP


In [59]:
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([x_mid[0], y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
v_k_history = []
v_horizon = []
acc_history = [0]
delta_history = [0]
psi_history = [0]
mean = [0]
is_turning_sharp = False
turning = []
sim_steps = 275
sim_steps = 300

# Continuous trajectory memory for warm-starting
last_sol_X = np.tile(x_current.reshape(4, 1), (1, n_horizon + 1))
last_sol_U = np.zeros((2, n_horizon))

for step in range(sim_steps):
    if step % 250 == 0 :print('step:',step)
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate Time-Varying Horizon Reference Profiles
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)
    #is_turning_sharp = (x_current[3] >= np.deg2rad(10)) or (x_current[3] <= -np.deg2rad(10))
    #is_turning_sharp = (abs(x_current[3]) >= np.deg2rad(5))
    #turning.append(is_turning_sharp)

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * 1.8:
            v_ref_horizon[k] = 0.0
        elif is_turning_sharp:
            v_ref_horizon[k] = 9
        else:
            v_ref_horizon[k] = 18

    # Update parameters
    opti.set_value(X0, x_current)
    opti.set_value(X_REF, x_ref_horizon)
    opti.set_value(Y_REF, y_ref_horizon)
    opti.set_value(V_REF, v_ref_horizon)
    

    # Seed guesses
    opti.set_initial(X, last_sol_X)
    opti.set_initial(U, last_sol_U)

    try:
        sol = opti.solve()
        u_control = sol.value(U[:, 0])
        last_sol_X = sol.value(X)
        last_sol_U = sol.value(U)
    except Exception as e:
        # Fallback: Retrieve the last non-optimal/sub-optimal values instead of crashing instantly
        u_control = opti.debug.value(U[:, 0])
        last_sol_X = opti.debug.value(X)
        last_sol_U = opti.debug.value(U)

    # --- Plant Simulator (Process Step using Forward Euler) ---
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    #is_turning_sharp = (abs(u_control[1]) >= np.deg2rad(15))
    is_turning_sharp = (u_control[1] >= np.deg2rad(5)) or (u_control[1] <= -np.deg2rad(5))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    v_k_history.append(last_sol_X[2])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])
    mean.append(np.mean(np.array([psi_history[-10:]])))
    v_horizon.append(v_ref_horizon)

    if s_total_traveled >= track_length and x_current[2] < 0.05:
        print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
        break

# Convert historical data into numpy arrays for rendering 

t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
v_history = np.array(v_history)
v_ref_history = np.array(v_ref_horizon)
acc_history = np.array(acc_history)
delta_history = np.array(delta_history)
delta_history = np.rad2deg(delta_history)

psi_history = np.array(psi_history)
psi_history = np.rad2deg(psi_history)
turning = [0 if t==False else 1 for t in turning]
turning = np.array(turning)
mean = np.array(mean)
mean = np.rad2deg(mean)

x_history2,y_history2 = RaceTrackOffset(x_history,y_history,3)

step: 0
step: 250


4. PLOT RESULTS

In [60]:
PlotMPCTracksPLY(x_history, y_history, x_history2, y_history2,
                  turning, v_history, x_mid=x_mid, y_mid=y_mid)




In [6]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_mid, y=y_mid, 
    mode='lines', 
    name='Track Midline Reference', 
    line=dict(color='lightgray', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=x_history, 
    y=y_history, 
    mode='lines+markers', 
    name='Vehicle Profile Path',
    line=dict(color='rgba(100,100,100,0.3)', width=2),
    marker=dict(
        size=5,
        color=turning,
        colorscale='Jet',
        cmin=np.min(turning),
        cmax=np.max(turning),
        colorbar=dict(
            title=dict(text="Speed (m/s)", side="bottom"),
            ticks="outside"
        ),
        showscale=True
    ),
    text=[f"Speed: {v:.2f} m/s" for v in v_history],
    hoverinfo='text+x+y'
))

fig.update_layout(
    title="Discrete MPC: Speed Profile Heatmap Across Lap Mission",
    xaxis_title="X Coordinate (meters)", 
    yaxis_title="Y Coordinate (meters)",
    yaxis=dict(scaleanchor="x", scaleratio=1), 
    showlegend=True,
    template="plotly_white"
)

fig.show()

In [ ]:
# %%
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *

def RaceTrackOffset(x, y, d=2.0):
    """
    Gera uma nova pista paralela onde cada ponto (x2, y2) dista exatamente
    'd' metros do ponto correspondente (x, y) original.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    
    nx = np.zeros(n)
    ny = np.zeros(n)
    
    for i in range(n):
        if i == 0:
            tx = x[1] - x[0]
            ty = y[1] - y[0]
        elif i == n - 1:
            tx = x[n-1] - x[n-2]
            ty = y[n-1] - y[n-2]
        else:
            tx = x[i+1] - x[i-1]
            ty = y[i+1] - y[i-1]
            
        norma = np.sqrt(tx**2 + ty**2)
        if norma == 0:
            norma = 1e-6
            
        tx /= norma
        ty /= norma
        
        nx[i] = -ty
        ny[i] = tx
        
    x_history2 = x + d * nx
    y_history2 = y + d * ny
    
    return x_history2, y_history2

# %% [markdown]
# 1. LOAD RACETRACK DATA & SPATIAL PARAMETRIZATION

# %%
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:] * size
y_mid = df['y_coords'].values[:] * size

dt = 0.1  # Control & Simulation evaluation step size (100ms)

if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

# %% [markdown]
# 2. MPC CONFIGURATION PARAMETERS
# %%
n_horizon = 3
l = 2.5  # Wheelbase

# Constraints limits
max_acc = 5.0
min_acc = -10.0
max_steer = np.deg2rad(30)
max_vel = 20.0
min_vel = 0.0

def get_linearized_dynamics(x_bar, u_bar, dt, l):
    """
    Returns the linearized state-space matrices A and B, plus the constant affine term C
    such that: X_{k+1} = A*X_k + B*U_k + C
    State order: [x, y, v, psi]
    Control order: [a, delta]
    """
    x, y, v, psi = x_bar
    a, delta = u_bar
    
    beta = np.arctan(0.5 * np.tan(delta))
    cos_angle = np.cos(psi + beta)
    sin_angle = np.sin(psi + beta)
    
    # Derivative of beta with respect to delta
    dbeta_ddelta = 0.5 * (1 + np.tan(delta)**2) / (1 + (0.5 * np.tan(delta))**2)
    
    # State transition matrix (A)
    A = np.eye(4)
    A[0, 2] = dt * cos_angle
    A[0, 3] = -dt * v * sin_angle
    A[1, 2] = dt * sin_angle
    A[1, 3] = dt * v * cos_angle
    A[3, 2] = dt * np.sin(beta) / l
    
    # Input matrix (B)
    B = np.zeros((4, 2))
    B[2, 0] = dt  # dv/da
    B[0, 1] = -dt * v * sin_angle * dbeta_ddelta
    B[1, 1] = dt * v * cos_angle * dbeta_ddelta
    B[3, 1] = dt * (v / l) * np.cos(beta) * dbeta_ddelta
    
    # Evaluation of non-linear dynamics at the operating point
    f_bar = np.array([
        x + dt * v * cos_angle,
        y + dt * v * sin_angle,
        v + dt * a,
        psi + dt * (v / l) * np.sin(beta)
    ])
    
    # Affine adjustment vector (C)
    C = f_bar - A @ x_bar - B @ u_bar
    return A, B, C

# %% [markdown]
# 3. CLOSED-LOOP SIMULATION LOOP
# %%
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([x_mid[0], y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
acc_history = [0.0]
delta_history = [0.0]
psi_history = [0.0]
turning = []
sim_steps = 400

# Continuous trajectory memory for warm-starting linearization profiles
last_u_control = np.array([0.0, 0.0])

for step in range(sim_steps):
    if step % 250 == 0: 
        print('step:', step)
        
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate reference tracking metrics
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)

    is_turning_sharp = (abs(last_u_control[1]) >= np.deg2rad(5))

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * 1.8:
            v_ref_horizon[k] = 0.0
        elif is_turning_sharp:
            v_ref_horizon[k] = 9.0
        else:
            v_ref_horizon[k] = 18.0

    # Linearize dynamics around the current state and last input execution
    A, B, C = get_linearized_dynamics(x_current, last_u_control, dt, l)

    # Define CVXPY tracking variables
    X_cvx = cp.Variable((4, n_horizon + 1))
    U_cvx = cp.Variable((2, n_horizon))

    cost = 0
    constraints = [X_cvx[:, 0] == x_current]

    for k in range(n_horizon):
        # LTV dynamics constraint evaluation
        constraints += [X_cvx[:, k+1] == A @ X_cvx[:, k] + B @ U_cvx[:, k] + C]
        
        # State Bounds
        constraints += [X_cvx[2, k] >= min_vel, X_cvx[2, k] <= max_vel]
        # Control Bounds
        constraints += [U_cvx[0, k] >= min_acc, U_cvx[0, k] <= max_acc]
        constraints += [U_cvx[1, k] >= -max_steer, U_cvx[1, k] <= max_steer]

        # Tracking cost implementation
        cost += cp.square(X_cvx[0, k] - x_ref_horizon[k])
        cost += cp.square(X_cvx[1, k] - y_ref_horizon[k])
        cost += 1.0 * cp.square(X_cvx[2, k] - v_ref_horizon[k])
        cost += 0.0125 * cp.square(U_cvx[0, k]) + 0.025 * cp.square(U_cvx[1, k])

    # Terminal state evaluation cost
    cost += cp.square(X_cvx[0, n_horizon] - x_ref_horizon[-1])
    cost += cp.square(X_cvx[1, n_horizon] - y_ref_horizon[-1])
    cost += 5.0 * cp.square(X_cvx[2, n_horizon] - v_ref_horizon[-1])

    # Instantiate problem setup & execute optimization via MOSEK
    prob = cp.Problem(cp.Minimize(cost), constraints)
    try:
        prob.solve(solver=cp.MOSEK)
        if prob.status not in ["optimal", "optimal_inaccurate"]:
            raise Exception("Solver could not find an optimal solution path.")
        u_control = U_cvx[:, 0].value
    except Exception as e:
        # Fallback Strategy: hold previous controls if system breaks
        u_control = last_u_control

    last_u_control = u_control

    # True Plant Simulator (Process Step using exact non-linear Forward Euler physics)
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    is_turning_sharp = (u_control[1] >= np.deg2rad(5)) or (u_control[1] <= -np.deg2rad(5))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])

    if s_total_traveled >= track_length and x_current[2] < 0.05:
        print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
        break

# Parse history sets
t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
v_history = np.array(v_history)
acc_history = np.array(acc_history)
delta_history = np.rad2deg(np.array(delta_history))
psi_history = np.rad2deg(np.array(psi_history))
turning = np.array([1 if t else 0 for t in turning])

x_history2, y_history2 = RaceTrackOffset(x_history, y_history, 3)

# %% [markdown]
# 4. PLOT RESULTS
# %%
PlotMPCTracksPLY(x_history, y_history, x_history2, y_history2,
                  turning, v_history, x_mid=x_mid, y_mid=y_mid)

# %%
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_mid, y=y_mid, mode='lines', name='Track Midline Reference', 
    line=dict(color='lightgray', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=x_history, y=y_history, mode='lines+markers', name='Vehicle Profile Path',
    line=dict(color='rgba(100,100,100,0.3)', width=2),
    marker=dict(
        size=5, color=turning, colorscale='Jet',
        colorbar=dict(title=dict(text="Turning State", side="bottom"), ticks="outside"),
        showscale=True
    ),
    text=[f"Speed: {v:.2f} m/s" for v in v_history],
    hoverinfo='text+x+y'
))

fig.update_layout(
    title="CVXPY LTV-MPC: Speed Profile Heatmap Across Lap Mission",
    xaxis_title="X Coordinate (meters)", yaxis_title="Y Coordinate (meters)",
    yaxis=dict(scaleanchor="x", scaleratio=1), template="plotly_white"
)
fig.show()

In [22]:
# %%
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *

def RaceTrackOffset(x, y, d=2.0):
    """
    Gera uma nova pista paralela onde cada ponto (x2, y2) dista exatamente
    'd' metros do ponto correspondente (x, y) original.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    
    nx = np.zeros(n)
    ny = np.zeros(n)
    
    for i in range(n):
        if i == 0:
            tx = x[1] - x[0]
            ty = y[1] - y[0]
        elif i == n - 1:
            tx = x[n-1] - x[n-2]
            ty = y[n-1] - y[n-2]
        else:
            tx = x[i+1] - x[i-1]
            ty = y[i+1] - y[i-1]
            
        norma = np.sqrt(tx**2 + ty**2)
        if norma == 0:
            norma = 1e-6
            
        tx /= norma
        ty /= norma
        
        nx[i] = -ty
        ny[i] = tx
        
    x_history2 = x + d * nx
    y_history2 = y + d * ny
    
    return x_history2, y_history2

# %% [markdown]
# 1. LOAD RACETRACK DATA & SPATIAL PARAMETRIZATION

# %%
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:] * size
y_mid = df['y_coords'].values[:] * size

dt = 0.1  # Control & Simulation evaluation step size (100ms)

if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

# %% [markdown]
# 2. MPC CONFIGURATION PARAMETERS & LINEARIZATION ENGINE

# %%
n_horizon = 3
l = 2.5  # Wheelbase

# Actuation and state bounds
max_acc = 5.0
min_acc = -10.0
max_steer = np.deg2rad(30)
max_vel = 20.0
min_vel = 0.0

def get_linearized_dynamics(x_bar, u_bar, dt, l):
    """
    Returns the linearized state-space matrices A and B, plus the constant affine term C
    such that: X_{k+1} = A*X_k + B*U_k + C
    """
    x, y, v, psi = x_bar
    a, delta = u_bar
    
    # Use a tiny virtual velocity if stopped to keep gradients alive
    v_lin = max(v, 1.0) 
    
    beta = np.arctan(0.5 * np.tan(delta))
    cos_angle = np.cos(psi + beta)
    sin_angle = np.sin(psi + beta)
    
    dbeta_ddelta = 0.5 * (1 + np.tan(delta)**2) / (1 + (0.5 * np.tan(delta))**2)
    
    # State transition matrix (A)
    A = np.eye(4)
    A[0, 2] = dt * cos_angle
    A[0, 3] = -dt * v_lin * sin_angle
    A[1, 2] = dt * sin_angle
    A[1, 3] = dt * v_lin * cos_angle
    A[3, 2] = dt * np.sin(beta) / l
    
    # Input matrix (B)
    B = np.zeros((4, 2))
    B[2, 0] = dt  
    B[0, 1] = -dt * v_lin * sin_angle * dbeta_ddelta
    B[1, 1] = dt * v_lin * cos_angle * dbeta_ddelta
    B[3, 1] = dt * (v_lin / l) * np.cos(beta) * dbeta_ddelta
    
    # Evaluate non-linear dynamics using the uniform linearization velocity projection
    f_bar = np.array([
        x + dt * v_lin * cos_angle,
        y + dt * v_lin * sin_angle,
        v_lin + dt * a,
        psi + dt * (v_lin / l) * np.sin(beta)
    ])
    
    # Affine adjustment vector (C)
    C = f_bar - A @ np.array([x, y, v_lin, psi]) - B @ u_bar
    return A, B, C

# MOSEK Solver Configurations
mosek_params = {
    "mosek.iparam.log": 0,
    "mosek.iparam.print_param": 0
}

# %% [markdown]
# 3. CLOSED-LOOP SIMULATION LOOP

# %%
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([x_mid[0], y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
acc_history = [0.0]
delta_history = [0.0]
psi_history = [0.0]
turning = []
sim_steps = 250

# Start with a tiny kickstart acceleration nudge to initialize state dynamics cleanly
last_u_control = np.array([1.0, 0.0])

for step in range(sim_steps):
    if step % 50 == 0: 
        print(f'step: {step} | Current Speed: {x_current[2]:.2f} m/s')
        
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate Time-Varying Horizon Reference Profiles
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)

    is_turning_sharp = (abs(last_u_control[1]) >= np.deg2rad(5))

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * 1.8:
            v_ref_horizon[k] = 0.0
        elif is_turning_sharp:
            v_ref_horizon[k] = 9.0
        else:
            v_ref_horizon[k] = 18.0

    # Linearize system matrices along current state constraints
    A, B, C = get_linearized_dynamics(x_current, last_u_control, dt, l)

    # Initialize CVXPY tracking variables
    X_cvx = cp.Variable((4, n_horizon + 1))
    U_cvx = cp.Variable((2, n_horizon))

    cost = 0
    constraints = [X_cvx[:, 0] == x_current]

    for k in range(n_horizon):
        constraints += [X_cvx[:, k+1] == A @ X_cvx[:, k] + B @ U_cvx[:, k] + C]
        
        # State & Control Bounds
        constraints += [X_cvx[2, k] >= min_vel, X_cvx[2, k] <= max_vel]
        constraints += [U_cvx[0, k] >= min_acc, U_cvx[0, k] <= max_acc]
        constraints += [U_cvx[1, k] >= -max_steer, U_cvx[1, k] <= max_steer]

        # Stage cost tracking metrics
        cost += cp.square(X_cvx[0, k] - x_ref_horizon[k])
        cost += cp.square(X_cvx[1, k] - y_ref_horizon[k])
        cost += 5.0 * cp.square(X_cvx[2, k] - v_ref_horizon[k])
        cost += 0.5 * cp.square(U_cvx[0, k]) + 0.5 * cp.square(U_cvx[1, k])

    # Terminal state evaluation cost setup
    cost += cp.square(X_cvx[0, n_horizon] - x_ref_horizon[-1])
    cost += cp.square(X_cvx[1, n_horizon] - y_ref_horizon[-1])
    cost += 5.0 * cp.square(X_cvx[2, n_horizon] - v_ref_horizon[-1])

    prob = cp.Problem(cp.Minimize(cost), constraints)
    
    # Robust execution block with automatic fallback
    try:
        prob.solve(solver=cp.MOSEK, mosek_params=mosek_params)
        if prob.status not in ["optimal", "optimal_inaccurate"] or U_cvx[:, 0].value is None:
            raise cp.SolverError("MOSEK issue detected.")
        u_control = U_cvx[:, 0].value
    except (cp.SolverError, Exception):
        # Automatic fallback to OSQP if MOSEK fails or has no valid license
        try:
            prob.solve(solver=cp.OSQP, verbose=False)
            u_control = U_cvx[:, 0].value if U_cvx[:, 0].value is None else U_cvx[:, 0].value
        except:
            u_control = last_u_control

    last_u_control = u_control

    # Exact Nonlinear Plant Simulator Execution Step (Forward Euler)
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    is_turning_sharp = (u_control[1] >= np.deg2rad(5)) or (u_control[1] <= -np.deg2rad(5))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])

    if s_total_traveled >= track_length and x_current[2] < 0.05:
        print(f"Simulation completed! Lap finished at step {step}.")
        break

# Parse history structures
t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
v_history = np.array(v_history)
acc_history = np.array(acc_history)
delta_history = np.rad2deg(np.array(delta_history))
psi_history = np.rad2deg(np.array(psi_history))
turning = np.array([1 if t else 0 for t in turning])

x_history2, y_history2 = RaceTrackOffset(x_history, y_history, 3)

# %% [markdown]
# 4. PLOT RESULTS
# %%
PlotMPCTracksPLY(x_history, y_history, x_history2, y_history2,
                  turning, v_history, x_mid=x_mid, y_mid=y_mid)

step: 0 | Current Speed: 0.00 m/s
step: 50 | Current Speed: 9.32 m/s
step: 100 | Current Speed: 9.05 m/s
step: 150 | Current Speed: 10.01 m/s
step: 200 | Current Speed: 13.51 m/s
